# SA-DMAE 전처리 노트북
BraTS 2021 3D NIfTI → 2.5D 슬라이스 `.pt` 파일 변환

**출력 형식:** `(3, 3, 224, 224)` float32 tensor  
**채널 순서:** T1ce / T2 / FLAIR  
**슬라이스:** (center_z-1, center_z, center_z+1) — 종양 ROI 기준

In [ ]:
# ── Cell 1: Google Drive 마운트 ───────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 2: 패키지 설치 ───────────────────────────────────────────────────
!pip install nibabel tqdm -q

In [ ]:
# ── Cell 3: 경로 설정 (여기만 수정) ──────────────────────────────────────

# BraTS2021 데이터 루트 (Google Drive 내 경로)
BRATS_ROOT = '/content/drive/MyDrive/BraTS2021'  # ← 실제 경로로 수정

# 전처리 결과 저장 경로 (Drive에 저장하므로 Colab 세션 종료 후에도 유지)
OUTPUT_DIR = '/content/drive/MyDrive/SA_DMAE_slices'  # ← 원하는 경로로 수정

# 슬라이스 설정
MODALITIES  = ['t1ce', 't2', 'flair']  # 채널 순서 고정
N_SLICES    = 3
TARGET_SIZE = (224, 224)

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
# ── Cell 4: Slicer 핵심 함수 ─────────────────────────────────────────────
import numpy as np
import nibabel as nib
import torch
import torch.nn.functional as F
from pathlib import Path


def load_volume(path):
    """NIfTI 파일 로드 → (H, W, D) float32 numpy array."""
    return nib.load(path).get_fdata(dtype=np.float32)


def normalize_volume(vol, low_pct=1.0, high_pct=99.0):
    """비영 voxel 기준 percentile 정규화 → [0, 1]."""
    nonzero = vol[vol > 0]
    if nonzero.size == 0:
        return vol
    lo = np.percentile(nonzero, low_pct)
    hi = np.percentile(nonzero, high_pct)
    if hi == lo:
        return np.zeros_like(vol)
    return (np.clip(vol, lo, hi) - lo) / (hi - lo)


def find_tumor_center_z(seg):
    """seg 마스크 기준 종양 Z-range 중앙 인덱스 반환."""
    tumor_mask = seg > 0
    z_with_tumor = np.where(tumor_mask.any(axis=(0, 1)))[0]
    if len(z_with_tumor) == 0:
        return seg.shape[2] // 2  # fallback: 볼륨 중앙
    return int(z_with_tumor[len(z_with_tumor) // 2])


def extract_slice_stack(volumes, center_z, n_slices=3, target_size=(224, 224)):
    """연속 n_slices 장 추출 → (n_slices, C, H, W) tensor."""
    half = n_slices // 2
    D = volumes[0].shape[2]
    z_indices = [min(max(center_z + o, 0), D - 1) for o in range(-half, half + 1)]

    slices = []
    for z in z_indices:
        channels = [vol[:, :, z] for vol in volumes]                    # list of (H,W)
        s = torch.from_numpy(np.stack(channels, axis=0)).unsqueeze(0)  # (1, C, H, W)
        s = F.interpolate(s, size=target_size, mode='bilinear', align_corners=False)
        slices.append(s.squeeze(0))                                     # (C, H, W)
    return torch.stack(slices, dim=0)                                   # (n_slices, C, H, W)


def process_case(case_dir, case_id, modalities, n_slices, target_size):
    """한 케이스를 처리하여 (n_slices, C, H, W) tensor 반환."""
    # 모달리티 로드 & 정규화
    volumes = [
        normalize_volume(load_volume(os.path.join(case_dir, f'{case_id}_{m}.nii.gz')))
        for m in modalities
    ]
    # seg 로드 → 종양 center Z
    seg = load_volume(os.path.join(case_dir, f'{case_id}_seg.nii.gz'))
    center_z = find_tumor_center_z(seg)

    return extract_slice_stack(volumes, center_z, n_slices, target_size), center_z


print('Slicer functions ready.')

In [ ]:
# ── Cell 5: 전체 데이터셋 전처리 메인 루프 ────────────────────────────────
from tqdm import tqdm

brats_root  = Path(BRATS_ROOT)
output_dir  = Path(OUTPUT_DIR)

# 전체 케이스 목록
all_cases = sorted([d for d in brats_root.iterdir() if d.is_dir()])
print(f'Total cases found: {len(all_cases)}')

# 이미 처리된 케이스 스킵 (재실행 시 이어서 진행)
done = {p.stem for p in output_dir.glob('*.pt')}
cases_to_run = [c for c in all_cases if c.name not in done]
print(f'Already done : {len(done)}  |  Remaining : {len(cases_to_run)}')

errors = []
log_path = output_dir / 'preprocess_log.txt'

for case_dir in tqdm(cases_to_run, desc='Preprocessing'):
    case_id = case_dir.name
    try:
        # 필요한 파일이 모두 있는지 확인
        required = [f'{case_id}_{m}.nii.gz' for m in MODALITIES] + [f'{case_id}_seg.nii.gz']
        missing  = [f for f in required if not (case_dir / f).exists()]
        if missing:
            raise FileNotFoundError(f'Missing: {missing}')

        stack, center_z = process_case(
            str(case_dir), case_id, MODALITIES, N_SLICES, TARGET_SIZE
        )

        # 저장: (3, 3, 224, 224) float32
        torch.save(stack.float(), output_dir / f'{case_id}.pt')

        # 로그 기록
        with open(log_path, 'a') as f:
            f.write(f'OK  {case_id}  center_z={center_z}  shape={list(stack.shape)}\n')

    except Exception as e:
        errors.append((case_id, str(e)))
        with open(log_path, 'a') as f:
            f.write(f'ERR {case_id}  {e}\n')

print(f'\nDone. Success: {len(cases_to_run) - len(errors)}  |  Errors: {len(errors)}')
if errors:
    print('Failed cases:')
    for cid, err in errors:
        print(f'  {cid}: {err}')

In [ ]:
# ── Cell 6: 결과 검증 ────────────────────────────────────────────────────
import random
import matplotlib.pyplot as plt

pt_files = sorted(output_dir.glob('*.pt'))
print(f'Total .pt files saved: {len(pt_files)}')

# 랜덤 샘플 3개 shape/range 확인
samples = random.sample(pt_files, min(3, len(pt_files)))
for p in samples:
    t = torch.load(p, map_location='cpu')
    print(f'{p.stem}  shape={list(t.shape)}  min={t.min():.3f}  max={t.max():.3f}')

# 샘플 1개 시각화 (n_slices=3, 채널=T1ce/T2/FLAIR)
sample = torch.load(samples[0], map_location='cpu')  # (3, 3, 224, 224)
mod_names   = ['T1ce', 'T2', 'FLAIR']
slice_names = ['prev (Z-1)', 'center (Z)', 'next (Z+1)']

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
fig.suptitle(f'Sample: {samples[0].stem}', fontsize=13)
for row, mod in enumerate(mod_names):
    for col, sname in enumerate(slice_names):
        ax = axes[row, col]
        ax.imshow(sample[col, row].numpy(), cmap='gray', vmin=0, vmax=1)
        ax.set_title(f'{mod} | {sname}', fontsize=8)
        ax.axis('off')
plt.tight_layout()
plt.savefig(str(output_dir / 'sample_verify.png'), dpi=100)
plt.show()
print('Verification complete.')